# 2. Análisis Exploratorio de Datos (EDA)

**Punto de partida:** el dataset crudo descargado en `1_GetData.ipynb` (`../data/WA_Fn-UseC_-Telco-Customer-Churn.csv`).

En inglés se conoce como **Exploratory Data Analysis (EDA)**; según [IBM](https://www.ibm.com/think/topics/exploratory-data-analysis), es la fase del proceso analítico en la que buscamos comprender los datos antes de limpiarlos, transformarlos o construir modelos predictivos. El objetivo principal no es obtener conclusiones definitivas, sino formular hipótesis, detectar problemas de calidad y entender el comportamiento general de la información.

### ¿Por qué es importante?
En proyectos reales, la mayor parte de los errores no proviene del algoritmo, sino de un entendimiento insuficiente de los datos y del contexto de negocio.

* Identificar datos faltantes.
* Detectar valores atípicos.
* Comprender la distribución de las variables.
* Encontrar relaciones entre variables.
* Verificar supuestos iniciales.
* Priorizar variables relevantes para el modelado.

**Entrega de este notebook:** ningún archivo nuevo — el EDA es puramente diagnóstico. Los hallazgos (nulos en `TotalCharges`, variables candidatas a transformar, correlaciones) se usan como insumo de decisiones en `3_FeatureEngineering.ipynb`.

Se importan `pandas` para manipulación de datos y `seaborn`/`matplotlib` para las visualizaciones univariadas y bivariadas que se construyen a lo largo del notebook.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

## cargar librerias de visualizacion
import seaborn as sns
import matplotlib.pyplot as plt

## Cargar datos

Se lee el dataset crudo generado por `1_GetData.ipynb`, sin ningún parámetro adicional de limpieza, para observar los datos exactamente como llegan de la fuente.

In [ ]:
fName = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

data = pd.read_csv(
                    fName
                    # , sep=",", na_values=[""," ","-","NA"]
                )
print("Dimensiones del Dataframe:", data.shape)
data.head()

## Análisis Exploratorio de Datos

Se trabaja sobre una **copia** (`df`) del dataset original (`data`), buena práctica que permite conservar siempre disponible la versión cruda por si se necesita comparar o reiniciar el análisis.

In [ ]:
## creando una copia
df = data.copy()
print("Dimensiones del Dataframe:", df.shape)
df.head()

Se fija explícitamente el nombre de la variable objetivo en una sola variable (`target`), para no repetir el string `"Churn"` en el resto del notebook y facilitar cambios futuros.

In [ ]:
# La variable target que corresponde a el comportamiento esperado.
target = "Churn"

### Revisión de tipos de datos

`df.info()` es el primer chequeo estructural: número de registros, nulos por columna y tipo de dato asignado por pandas. Aquí se detecta la primera inconsistencia: `TotalCharges` debería ser numérica, pero pandas la interpretó como texto (`object`).

In [ ]:
df.info()

Se intenta convertir `TotalCharges` a numérico. El error que arroja esta celda **es intencional**: confirma que existen valores no convertibles directamente (no son nulos vacíos, sino texto no numérico, típicamente espacios en blanco).

In [ ]:
pd.to_numeric(df["TotalCharges"])

Se usa `errors="coerce"` para identificar puntualmente cuáles registros no se pueden convertir (quedan como `NaN`), y se filtran para inspeccionarlos.

In [ ]:
## validando por que no se puede convertir
df[
    pd.to_numeric(df["TotalCharges"], errors="coerce").isnull()
]

Se aplica la conversión definitiva con `errors="coerce"`: los valores no numéricos se convierten en `NaN` (nulos), y ahora se revisa nuevamente `df.info()` para confirmar que `TotalCharges` ya quedó como tipo numérico (`float64`).

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.info()

### Revisión de cardinalidad

Se cuenta el número de valores únicos por columna. Esto ayuda a distinguir variables categóricas (pocos valores distintos) de variables continuas o identificadoras (muchos valores distintos, potencialmente únicos por registro).

In [ ]:
print("Revision de datos unicos por variable:")
df.nunique()

`customerID` tiene tantos valores únicos como registros tiene el dataset, lo que confirma que es un identificador único por cliente. Por lo tanto, se puede usar como índice del `DataFrame` en lugar de tratarlo como una variable analítica más.

In [ ]:
df.set_index("customerID", inplace=True)
df.head()

### Distribución de variables categóricas / de baja cardinalidad

Para las variables con 10 o menos valores distintos (típicamente categóricas o binarias), se revisa la proporción de cada categoría. Esto permite detectar clases dominantes, categorías raras o posibles errores de captura antes de pasar al análisis gráfico.

In [ ]:
print("Distribucion de valores por cada variable (con una cardinalidad <=10):")
cols = df.nunique()[df.nunique()<=10].index
for c in cols:
    print(f"\t{c:<16}", [f"{k}: {v:,.1%}" for k,v in dict(df[c].value_counts(normalize=True)).items()])

In [ ]:
print("Datos estadisticos para variables numericas:")
df.describe().T

### Valores nulos

Ante registros con valores nulos o vacíos, las alternativas típicas son:
* Imputar (reemplazar por un valor estimado).
* Eliminar el registro.

La decisión correcta depende de cuántos registros están afectados y si su eliminación introduce sesgo en la variable objetivo.

In [ ]:
print("Registros con nulos:")
filtro_nulos = df.isnull().sum(axis=1)>0
df[filtro_nulos]

In [ ]:
print("Revision de nulos:")
nulos = df.isnull().sum()
pd.concat([nulos[nulos>0], round(nulos[nulos>0]/len(df)*100,2)], axis=1).rename(columns={0:"Nulos", 1:"%"})

In [ ]:
print(f"Distribucion de los nulos con respecto al {target=}: {dict(df[filtro_nulos][target].value_counts())}")
print(f"Distribucion del target:", dict(df[target].value_counts()))

**Decisión:** la opción más favorable en este caso es eliminar los registros, ya que representan una fracción mínima del dataset (0.16% del total) y además pertenecen a la clase mayoritaria del target, por lo que su eliminación no introduce un sesgo relevante en la variable objetivo.

In [ ]:
df = df[~filtro_nulos]
print("Dimensiones del Dataframe:", df.shape)
df.head()

In [ ]:
print(f"items nulos = {df.isnull().sum().sum()}")

### Clasificación de variables: numéricas vs. categóricas

Separar explícitamente las variables numéricas de las categóricas facilita aplicar el tipo de análisis (y de transformación, en el módulo siguiente) adecuado a cada grupo.

In [ ]:
# Variables numericas
cols_num = df.select_dtypes(["int","float"]).columns
df[cols_num]

In [ ]:
cols_cat = df.drop(cols_num, axis=1).columns
# cols_cat = df.select_dtypes(["str","object"]).columns
df[cols_cat]

## Estadísticas descriptivas

Solo aplican directamente sobre las variables numéricas: medidas de tendencia central (media, mediana) y de dispersión (desviación estándar, rango), útiles para detectar asimetrías o valores atípicos.

In [ ]:
df.describe().T

### Análisis univariado

Se revisa la distribución individual de las variables más relevantes, comenzando por la variable objetivo.

In [ ]:
sns.countplot(data=df, x="Churn")
plt.title("Distribución de Churn")
plt.show()

Distribución de la antigüedad del cliente (`tenure`, en meses).

In [ ]:
sns.histplot(df["tenure"], bins=30, kde=True)
plt.title("Distribución de la antigüedad (tenure)")
plt.show()

Distribución de los cargos mensuales (`MonthlyCharges`).

In [ ]:
sns.histplot(df["MonthlyCharges"], bins=30, kde=True)
plt.title("Distribución de cargos mensuales")
plt.show()

Misma variable, pero normalizando el histograma como **densidad de probabilidad** (`stat="probability"`) en lugar de conteos absolutos. Esto es útil cuando se quiere comparar la forma de la distribución independientemente del tamaño de la muestra, o compararla más adelante contra otra variable con distinta escala de conteo.

In [ ]:
sns.histplot(df["MonthlyCharges"], bins=30, kde=True, stat="probability")
plt.title("Distribución de cargos mensuales")
plt.show()

Distribución de los cargos totales acumulados (`TotalCharges`).

In [ ]:
sns.histplot(df["TotalCharges"], bins=30, kde=True)
plt.title("Distribución de cargos totales")
plt.show()

### Análisis bivariado: relación con la variable objetivo

A partir de aquí se cruzan las variables con `Churn`, para identificar diferencias de comportamiento entre clientes que abandonan y los que permanecen.

In [ ]:
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure según Churn")
plt.show()

In [ ]:
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("MonthlyCharges según Churn")
plt.show()

Relación conjunta entre antigüedad y cargo mensual, coloreada por `Churn`, para observar visualmente si existen zonas del espacio con mayor concentración de abandono.

In [ ]:
sns.scatterplot(data=df, x="tenure", y="MonthlyCharges", alpha=0.4, hue=target)
plt.title("tenure vs MonthlyCharges")
plt.show()

Proporción de churn según el tipo de contrato (tabla de contingencia normalizada por fila).

In [ ]:
pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100

In [ ]:
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Churn por tipo de contrato")
plt.show()

Distribución de churn según método de pago.

In [ ]:
sns.countplot(data=df, x="PaymentMethod", hue="Churn")
plt.xticks(rotation=20)
plt.title("Churn por método de pago")
plt.show()

Distribución de churn según tipo de servicio de internet contratado.

In [ ]:
sns.countplot(data=df, x="InternetService", hue="Churn")
plt.title("Churn por servicio de internet")
plt.show()

### Correlación entre variables numéricas

La matriz de correlación ayuda a detectar relaciones lineales fuertes entre variables numéricas, útil tanto para entender el negocio como para anticipar posible redundancia de variables en el modelado.

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[cols_num].corr(), annot=True, cmap="coolwarm", fmt=".1%")
plt.title("Matriz de correlación")
plt.show()

### Análisis exploratorio generalizado por variable

En lugar de repetir manualmente el mismo gráfico para cada variable, se automatiza el recorrido: para cada columna categórica se calcula la proporción de churn por categoría, y para cada columna numérica se compara su distribución entre las dos clases del target. Esto agiliza detectar rápidamente qué variables muestran mayor asociación aparente con el abandono.

### Vista previa de una sola variable antes de generalizar

Antes de automatizar el recorrido por todas las variables (siguiente celda), conviene revisar el resultado sobre **una sola variable conocida** (`MonthlyCharges`) para confirmar que la lógica es la esperada: comparar estadísticas descriptivas y distribución entre clientes con y sin `Churn`.

In [ ]:
c="MonthlyCharges"

In [ ]:
distribucion_target = df[[target, c]].groupby(target).describe()
distribucion_target.columns = [x[1] for x in distribucion_target.columns]
distribucion_target

In [ ]:
sns.boxplot(data=df, hue=target, y=c)

**Nota:** las dos celdas siguientes generalizan a *todas* las variables la misma lógica que acabas de ver para `MonthlyCharges` (comparar la distribución de cada variable entre clientes con y sin `Churn`), recorriendo las columnas con un bucle `for`. No necesitas entender el código línea por línea — concéntrate en leer los gráficos que produce, uno por variable.

In [ ]:
for c,t in df.drop(target, axis=1).dtypes.items():
    if t.name in ["str","object"]:
        distribucion_target = pd.crosstab(index=df[c], columns=df[target], values=target, aggfunc="count", normalize="index")
        distribucion_target = distribucion_target.unstack().reset_index().rename(columns={0:"proportion"})

        plt.figure(figsize=(4,3))
        sns.barplot(data=distribucion_target, x=c, y="proportion", hue=target)
        plt.title(f"Distribución de {target} por [{c}]", size=9)
        plt.legend(loc="upper right", fontsize=8)
        plt.xticks(rotation=20, fontsize=8)
        plt.xlabel(None)
        plt.show()
    else:
        plt.figure(figsize=(4,3))
        sns.boxplot(data=df, x=target, hue=target, y=c)
        plt.title(f"Distribución de {target} por [{c}]", size=9)
        plt.xticks(rotation=20, fontsize=8)
        plt.xlabel(None)
        plt.show()


In [ ]:
for c,t in df.drop(target, axis=1).dtypes.items():
    if t.name in ["str","object"]:
        pass
    else:
        plt.figure(figsize=(5,3))
        sns.histplot(data=df, x=c, hue=target, kde=True, alpha=0.4)
        plt.title(f"Distribución de [{c}] por {target=}", size=9)
        # plt.xticks(rotation=20, fontsize=8)
        plt.xlabel(None)
        plt.show()

## Preguntas orientadoras del EDA

* ¿Qué características tienen los clientes que abandonan la compañía?
* ¿Cómo se comporta la antigüedad de los clientes?
* ¿Qué servicios son más frecuentes?
* ¿Existen diferencias en los cargos mensuales?
* ¿Qué métodos de pago predominan?
* ¿Hay variables con posibles problemas de calidad?

## Resultados esperados del EDA:

1. Comprensión de la estructura del dataset
    * Número de filas y columnas.
    * Tipos de datos.
    * Variables categóricas y numéricas.
    * Variable objetivo (Churn).

2. Calidad inicial de los datos
    * Valores nulos o vacíos.
    * Duplicados.
    * Inconsistencias aparentes.
    * Variables importadas con tipo incorrecto.

3. Análisis univariado
    * Distribuciones numéricas.
    * Frecuencias categóricas.
    * Medidas descriptivas básicas.

4. Análisis bivariado
    * Relación entre variables y Churn.
    * Comparación de grupos.
    * Tendencias y asociaciones.

5. Hallazgos preliminares
    * Patrones observados.
    * Variables potencialmente relevantes.
    * Riesgos o limitaciones de los datos.

<figure>
  <blockquote>
    "El Analisis de Datos Exploratorio (EDA) es el puente entre el conocimiento del negocio y el modelamiento analítico."
  </blockquote>
  <figcaption>— <cite>Diego Herrera Malambo.</cite></figcaption>
</figure>

### Siguiente paso

Los hallazgos de este EDA (nulos en `TotalCharges` asociados a clientes nuevos, variables categóricas pendientes de codificar, correlaciones relevantes con el target) se retoman y se resuelven de forma sistemática en `3_FeatureEngineering.ipynb`, que parte nuevamente del dataset crudo para dejarlo listo para modelado.